In [24]:
#all imports
import pandas as pd
import numpy as np
from pathlib import Path

# Set the actual location of your CyberGuard project
PROJECT_DIR = Path(r"C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD")

DATA_DIR = PROJECT_DIR / "malicious_url" / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "malicious_url" / "models"
RESULTS_DIR = PROJECT_DIR / "malicious_url" / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

phiusiil_path = DATA_DIR / "phiusiil_clean.csv"
uci_path = DATA_DIR / "uci_clean.csv"

phiusiil_df = pd.read_csv(phiusiil_path)
uci_df = pd.read_csv(uci_path)

print("PhiUSIIL shape:", phiusiil_df.shape)
print("UCI shape:", uci_df.shape)
print("PhiUSIIL columns:", phiusiil_df.columns.tolist())
print("UCI columns:", uci_df.columns.tolist())

PhiUSIIL shape: (235370, 57)
UCI shape: (11055, 33)
PhiUSIIL columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label', 'cyberguard

In [25]:
#phisUSIL features group
from sklearn.model_selection import train_test_split

phi_exclude = [
    "FILENAME",
    "URL",
    "Domain",
    "TLD",
    "Title",
    "label",
    "cyberguard_label"
]

X_phi = phiusiil_df.drop(
    columns=phi_exclude,
    errors="ignore"
)

X_phi = X_phi.select_dtypes(include=np.number)
y_phi = phiusiil_df["cyberguard_label"].astype(int)

X_phi_train, X_phi_test, y_phi_train, y_phi_test = train_test_split(
    X_phi,
    y_phi,
    test_size=0.20,
    random_state=42,
    stratify=y_phi
)

print("Training:", X_phi_train.shape)
print("Testing:", X_phi_test.shape)

Training: (188296, 50)
Testing: (47074, 50)


In [26]:
#UCI features group
uci_exclude = [
    "id",
    "Result",
    "cyberguard_label"
]

X_uci = uci_df.drop(
    columns=uci_exclude,
    errors="ignore"
)

X_uci = X_uci.select_dtypes(include=np.number)
y_uci = uci_df["cyberguard_label"].astype(int)

X_uci_train, X_uci_test, y_uci_train, y_uci_test = train_test_split(
    X_uci,
    y_uci,
    test_size=0.20,
    random_state=42,
    stratify=y_uci
)

print("Training:", X_uci_train.shape)
print("Testing:", X_uci_test.shape)

Training: (8844, 30)
Testing: (2211, 30)


In [27]:
experiments = [
    (
        "PhiUSIIL",
        X_phi_train,
        X_phi_test,
        y_phi_train,
        y_phi_test
    ),
    (
        "UCI",
        X_uci_train,
        X_uci_test,
        y_uci_train,
        y_uci_test
    )
]

print("Experiments:", len(experiments))
print("Datasets:", [exp[0] for exp in experiments])

Experiments: 2
Datasets: ['PhiUSIIL', 'UCI']


In [28]:
#training and comparing two models
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    results = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1_score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) else 0
        ),
        "false_negative_rate": (
            fn / (fn + tp) if (fn + tp) else 0
        )
    }

    print(f"\n===== {name} =====")

    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Legitimate", "Malicious"],
        zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ))

    return results, model

In [29]:
print("Evaluation function cell executed successfully.")
print(evaluate_model)

Evaluation function cell executed successfully.
<function evaluate_model at 0x0000018F6ABA9080>


In [30]:
print("Experiments:", len(experiments))
print("Datasets:", [exp[0] for exp in experiments])

Experiments: 2
Datasets: ['PhiUSIIL', 'UCI']


In [31]:
results = []
trained_models = {}

for dataset_name, X_train, X_test, y_train, y_test in experiments:

    print(f"\nStarting dataset: {dataset_name}", flush=True)

    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=150,
            random_state=42
        )
    }

    for model_name, model in models.items():

        print(f"Training {model_name}...", flush=True)

        result, trained_model = evaluate_model(
            model,
            X_train,
            X_test,
            y_train,
            y_test,
            f"{dataset_name}_{model_name}"
        )

        results.append({
            "dataset": dataset_name,
            **result
        })

        trained_models[f"{dataset_name}_{model_name}"] = trained_model

        print(f"Finished {model_name}", flush=True)

results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON")
display(results_df.sort_values("f1_score", ascending=False))


Starting dataset: PhiUSIIL
Training RandomForest...

===== PhiUSIIL_RandomForest =====
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     26970
   Malicious       1.00      1.00      1.00     20104

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074

Confusion Matrix:
[[26970     0]
 [    0 20104]]
Finished RandomForest
Training HistGradientBoosting...

===== PhiUSIIL_HistGradientBoosting =====
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     26970
   Malicious       1.00      1.00      1.00     20104

    accuracy                           1.00     47074
   macro avg       1.00      1.00      1.00     47074
weighted avg       1.00      1.00      1.00     47074

Confusion Matrix:
[[26970     0]
 [    0 20104]]
Finished HistGradientBoosting

Starting dataset: UCI
Training 

,dataset,model,accuracy,precision,recall,f1_score,false_positive_rate,false_negative_rate
0,PhiUSIIL,PhiUSIIL_RandomForest,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
1,PhiUSIIL,PhiUSIIL_HistGradientBoosting,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
2,UCI,UCI_RandomForest,0.976029,0.978328,0.967347,0.972807,0.017059,0.032653
3,UCI,UCI_HistGradientBoosting,0.971506,0.976116,0.959184,0.967576,0.018684,0.040816


In [1]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

PROJECT_DIR = Path(r"C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD")

DATA_PATH = (
    PROJECT_DIR / "malicious_url" / "data" /
    "processed" / "phiusiil_clean.csv"
)

SRC_DIR = PROJECT_DIR / "malicious_url" / "src"
MODEL_DIR = PROJECT_DIR / "malicious_url" / "models"
RESULTS_DIR = PROJECT_DIR / "malicious_url" / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from url_features import extract_url_features

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())
print("Label distribution:")
print(df["cyberguard_label"].value_counts(dropna=False))

Dataset shape: (235370, 57)
Available columns: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label', 'cyberguard_label']
Label distribu

In [2]:
url_only_features = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "NoOfLettersInURL",
    "LetterRatioInURL",
    "NoOfDegitsInURL",
    "DegitRatioInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL",
    "IsHTTPS",
]

missing = [
    col for col in ["URL", "cyberguard_label"]
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Features to generate:", len(url_only_features))

Features to generate: 18


In [6]:
url_values = df["URL"].astype("string").str.strip()
valid_url = url_values.notna() & url_values.ne("")
valid_label = df["cyberguard_label"].isin([0, 1])

work = df.loc[valid_url & valid_label].copy()
work["URL"] = url_values.loc[work.index]

feature_rows = []
failed_rows = []

for idx, url in work["URL"].items():
    try:
        features = extract_url_features(str(url))

        missing_features = set(url_only_features) - set(features)
        if missing_features:
            raise ValueError(
                f"Extractor missing features: {sorted(missing_features)}"
            )

        feature_rows.append({
            "source_index": idx,
            **{f: features[f] for f in url_only_features}
        })

    except Exception as exc:
        failed_rows.append({
            "source_index": idx,
            "URL": str(url),
            "error": str(exc)
        })

if not feature_rows:
    raise ValueError("No URL features were extracted successfully.")

feature_df = pd.DataFrame(feature_rows).set_index("source_index")
failed_df = pd.DataFrame(failed_rows)

y = work.loc[feature_df.index, "cyberguard_label"].astype(int)

print("Successfully extracted:", len(feature_df))
print("Failed:", len(failed_df))
print("Feature matrix shape:", feature_df.shape)
print("Label counts:")
print(y.value_counts())

if not failed_df.empty:
    display(failed_df.head(10))



Successfully extracted: 235370
Failed: 0
Feature matrix shape: (235370, 18)
Label counts:
cyberguard_label
0    134850
1    100520
Name: count, dtype: int64


In [7]:
print("Data types:")
print(feature_df.dtypes)

print("\nMissing values:", int(feature_df.isna().sum().sum()))
print(
    "Infinite values:",
    int(np.isinf(feature_df.to_numpy(dtype=float)).sum())
)

print("\nFeature ranges:")
display(feature_df.describe().T)

assert list(feature_df.columns) == url_only_features
assert not feature_df.isna().any().any()
assert np.isfinite(feature_df.to_numpy(dtype=float)).all()
assert y.isin([0, 1]).all()

Data types:
URLLength                       int64
DomainLength                    int64
IsDomainIP                      int64
TLDLength                       int64
NoOfSubDomain                   int64
HasObfuscation                  int64
NoOfObfuscatedChar              int64
ObfuscationRatio              float64
NoOfLettersInURL                int64
LetterRatioInURL              float64
NoOfDegitsInURL                 int64
DegitRatioInURL               float64
NoOfEqualsInURL                 int64
NoOfQMarkInURL                  int64
NoOfAmpersandInURL              int64
NoOfOtherSpecialCharsInURL      int64
SpacialCharRatioInURL         float64
IsHTTPS                         int64
dtype: object

Missing values: 0
Infinite values: 0

Feature ranges:


,count,mean,std,min,25%,50%,75%,max
URLLength,235370.0,35.339920,41.233000,14.000000,24.000000,28.000000,35.000000,6097.000000
DomainLength,235370.0,21.454523,9.126058,4.000000,16.000000,20.000000,24.000000,110.000000
IsDomainIP,235370.0,0.002575,0.050676,0.000000,0.000000,0.000000,0.000000,1.000000
TLDLength,235370.0,2.763751,0.597532,2.000000,2.000000,3.000000,3.000000,13.000000
NoOfSubDomain,235370.0,1.159744,0.602036,0.000000,1.000000,1.000000,1.000000,10.000000
HasObfuscation,235370.0,0.010588,0.102350,0.000000,0.000000,0.000000,0.000000,1.000000
NoOfObfuscatedChar,235370.0,0.025080,0.837185,0.000000,0.000000,0.000000,0.000000,149.000000
ObfuscationRatio,235370.0,0.000107,0.002386,0.000000,0.000000,0.000000,0.000000,0.294840
NoOfLettersInURL,235370.0,27.187318,28.684699,4.000000,19.000000,23.000000,28.000000,5196.000000
LetterRatioInURL,235370.0,0.777460,0.074324,0.142857,0.760000,0.793103,0.821429,0.944444


In [8]:
from urllib.parse import urlsplit

def get_hostname(url):
    value = str(url).strip()
    if "://" not in value:
        value = "//" + value

    try:
        return (urlsplit(value).hostname or "").lower().rstrip(".")
    except ValueError:
        return ""

groups = work.loc[feature_df.index, "URL"].map(get_hostname)

# Use a stable fallback for URLs where a hostname could not be parsed.
groups = groups.where(groups.ne(""), "unparsed_" + groups.index.astype(str))

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(feature_df, y, groups=groups)
)

X_train = feature_df.iloc[train_idx]
X_test = feature_df.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training domains:", len(train_groups))
print("Testing domains:", len(test_groups))
print("Domain overlap:", len(train_groups & test_groups))

assert len(train_groups & test_groups) == 0

Training rows: 187856
Testing rows: 47514
Training domains: 176061
Testing domains: 44016
Domain overlap: 0


In [9]:
url_rf_consistent = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

url_rf_consistent.fit(X_train, y_train)

predictions = url_rf_consistent.predict(X_test)

tn, fp, fn, tp = confusion_matrix(
    y_test, predictions, labels=[0, 1]
).ravel()

metrics = {
    "accuracy": accuracy_score(y_test, predictions),
    "precision_malicious": precision_score(
        y_test, predictions, zero_division=0
    ),
    "recall_malicious": recall_score(
        y_test, predictions, zero_division=0
    ),
    "f1_malicious": f1_score(
        y_test, predictions, zero_division=0
    ),
    "false_positive_rate": fp / (fp + tn) if fp + tn else 0,
    "false_negative_rate": fn / (fn + tp) if fn + tp else 0,
}

print("Domain-held-out evaluation:")
for name, value in metrics.items():
    print(f"{name}: {value:.6f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    target_names=["Legitimate", "Malicious"],
    zero_division=0
))

Domain-held-out evaluation:
accuracy: 0.995012
precision_malicious: 0.996616
recall_malicious: 0.991802
f1_malicious: 0.994203
false_positive_rate: 0.002553
false_negative_rate: 0.008198

Confusion matrix:
[[26953    69]
 [  168 20324]]

Classification report:
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      1.00     27022
   Malicious       1.00      0.99      0.99     20492

    accuracy                           1.00     47514
   macro avg       1.00      0.99      0.99     47514
weighted avg       1.00      1.00      1.00     47514



In [10]:
import json
import joblib

MODEL_PATH = (
    MODEL_DIR / "cyberguard_url_random_forest_consistent.pkl"
)
FEATURE_PATH = MODEL_DIR / "url_only_features_consistent.json"
METRICS_PATH = RESULTS_DIR / "url_consistent_model_metrics.json"

joblib.dump(url_rf_consistent, MODEL_PATH)

with open(FEATURE_PATH, "w", encoding="utf-8") as f:
    json.dump(url_only_features, f, indent=4)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print("Saved model:", MODEL_PATH)
print("Saved feature schema:", FEATURE_PATH)
print("Saved metrics:", METRICS_PATH)

Saved model: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\models\cyberguard_url_random_forest_consistent.pkl
Saved feature schema: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\models\url_only_features_consistent.json
Saved metrics: C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\results\url_consistent_model_metrics.json


In [11]:
loaded_model = joblib.load(MODEL_PATH)

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    loaded_features = json.load(f)

assert loaded_features == url_only_features

test_row = X_test.iloc[[0]]
prediction = loaded_model.predict(test_row)[0]

print("Test URL:", work.loc[test_row.index[0], "URL"])
print("Prediction:", "Malicious" if prediction == 1 else "Legitimate")
print("True label:", "Malicious" if y_test.iloc[0] == 1 else "Legitimate")
print("Reload verification: passed")

Test URL: https://www.sfnmjournal.com
Prediction: Legitimate
True label: Legitimate
Reload verification: passed


In [13]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD")
MODEL_DIR = PROJECT_DIR / "malicious_url" / "models"

MODEL_PATH = MODEL_DIR / "cyberguard_url_random_forest_consistent.pkl"
FEATURE_PATH = MODEL_DIR / "url_only_features_consistent.json"

print("Model exists:", MODEL_PATH.exists(), MODEL_PATH)
print("Feature schema exists:", FEATURE_PATH.exists(), FEATURE_PATH)

Model exists: True C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\models\cyberguard_url_random_forest_consistent.pkl
Feature schema exists: True C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD\malicious_url\models\url_only_features_consistent.json


In [15]:
import sys
import json
import joblib
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\HP\OneDrive\Desktop\BPUT\CYBERGUARD")
SRC_DIR = PROJECT_DIR / "malicious_url" / "src"
MODEL_DIR = PROJECT_DIR / "malicious_url" / "models"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from url_features import extract_url_features

MODEL_PATH = MODEL_DIR / "cyberguard_url_random_forest_consistent.pkl"
FEATURE_PATH = MODEL_DIR / "url_only_features_consistent.json"

model = joblib.load(MODEL_PATH)

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    feature_names = json.load(f)


def test_url(url):
    print("\nURL:", url)

    try:
        features = extract_url_features(url)

        missing = set(feature_names) - set(features)
        if missing:
            raise ValueError(f"Missing features: {sorted(missing)}")

        X = pd.DataFrame(
            [[features[name] for name in feature_names]],
            columns=feature_names
        )

        prediction = int(model.predict(X)[0])
        scores = model.predict_proba(X)[0]
        classes = list(model.classes_)

        malicious_score = float(scores[classes.index(1)])

        print(
            "Prediction:",
            "MALICIOUS" if prediction == 1 else "LEGITIMATE"
        )
        print(f"Malicious model score: {malicious_score:.2%}")

        return {
            "url": url,
            "prediction": prediction,
            "malicious_score": malicious_score
        }

    except Exception as exc:
        print("ERROR:", exc)
        return None


# Smoke tests: these are examples, not guaranteed classifications.
test_urls = [
    "https://www.wikipedia.org",
    "https://www.python.org",
    "http://192.0.2.1/login",
    "https://example.com/%2e%2e/login",
]

results = [test_url(url) for url in test_urls]


URL: https://www.wikipedia.org
Prediction: LEGITIMATE
Malicious model score: 0.26%

URL: https://www.python.org
Prediction: LEGITIMATE
Malicious model score: 0.28%

URL: http://192.0.2.1/login
Prediction: MALICIOUS
Malicious model score: 100.00%

URL: https://example.com/%2e%2e/login
Prediction: MALICIOUS
Malicious model score: 100.00%


In [16]:
import pandas as pd

DATA_PATH = (
    PROJECT_DIR / "malicious_url" / "data" /
    "processed" / "phiusiil_clean.csv"
)

df = pd.read_csv(DATA_PATH)

# Select 5 examples from each class.
samples = pd.concat([
    df[df["cyberguard_label"] == 0].sample(
        n=5, random_state=10
    ),
    df[df["cyberguard_label"] == 1].sample(
        n=5, random_state=10
    )
])

test_results = []

for _, row in samples.iterrows():
    result = test_url(str(row["URL"]))

    if result is not None:
        actual = int(row["cyberguard_label"])

        test_results.append({
            "URL": row["URL"],
            "Actual": actual,
            "Predicted": result["prediction"],
            "Correct": actual == result["prediction"],
            "Malicious model score": result["malicious_score"]
        })

test_df = pd.DataFrame(test_results)
display(test_df)

if not test_df.empty:
    print(
        "Sample accuracy:",
        f"{test_df['Correct'].mean():.2%}"
    )


URL: https://www.edarabia.com
Prediction: LEGITIMATE
Malicious model score: 0.35%

URL: https://www.revolutiontea.com
Prediction: LEGITIMATE
Malicious model score: 0.38%

URL: https://www.paulstolper.com
Prediction: LEGITIMATE
Malicious model score: 0.58%

URL: https://www.leg.state.nv.us
Prediction: LEGITIMATE
Malicious model score: 0.00%

URL: https://www.polsl.pl
Prediction: LEGITIMATE
Malicious model score: 0.36%

URL: http://www.compress.to
Prediction: MALICIOUS
Malicious model score: 100.00%

URL: https://fb-restriction-case-a8103.firebaseapp.com/
Prediction: MALICIOUS
Malicious model score: 100.00%

URL: http://www.st.cloins.com
Prediction: MALICIOUS
Malicious model score: 100.00%

URL: https://bejadin.sa/outlook/index.html
Prediction: MALICIOUS
Malicious model score: 100.00%

URL: http://www.iacademygroup.cl
Prediction: MALICIOUS
Malicious model score: 100.00%


,URL,Actual,Predicted,Correct,Malicious model score
0,https://www.edarabia.com,0,0,True,0.003503
1,https://www.revolutiontea.com,0,0,True,0.003760
2,https://www.paulstolper.com,0,0,True,0.005786
3,https://www.leg.state.nv.us,0,0,True,0.000000
4,https://www.polsl.pl,0,0,True,0.003635
5,http://www.compress.to,1,1,True,1.000000
6,https://fb-restriction-case-a8103.firebaseapp....,1,1,True,1.000000
7,http://www.st.cloins.com,1,1,True,1.000000
8,https://bejadin.sa/outlook/index.html,1,1,True,1.000000
9,http://www.iacademygroup.cl,1,1,True,1.000000


Sample accuracy: 100.00%
